# Sandbox 完整指南

`SandboxExecutor` 是 ADLab 中用于安全执行代码的组件。它在隔离的子进程中运行代码，提供超时保护和资源限制。

## 目录
1. [基本概念](#1-基本概念)
2. [初始化参数](#2-初始化参数)
3. [执行方法](#3-执行方法)
4. [使用示例](#4-使用示例)

## 1. 基本概念

In [ ]:
# Sandbox 的核心特性:
# 1. 进程隔离 - 代码在独立进程中运行
# 2. 超时保护 - 防止无限循环
# 3. 资源限制 - 控制内存使用
# 4. 输出重定向 - 捕获或丢弃输出

from algodisco.toolkit.sandbox import SandboxExecutor
from algodisco.base.evaluator import EvalResult

## 2. 初始化参数

In [ ]:
# SandboxExecutor 初始化参数:

# evaluate_worker: 要执行的工作对象（必须有可调用的方法）
# recur_kill_eval_proc: 是否递归终止子进程
# debug_mode: 是否启用调试模式
# join_timeout_seconds: 等待进程完成的时间（秒）
# shm_max_size_bytes: 共享内存最大大小（默认20MB）

print("SandboxExecutor 参数说明:")
print("- evaluate_worker: 必需，执行的工作对象")
print("- recur_kill_eval_proc: 可选，是否递归终止子进程")
print("- debug_mode: 可选，是否启用调试")
print("- join_timeout_seconds: 可选，默认10秒")
print("- shm_max_size_bytes: 可选，默认20MB")

## 3. 执行方法

In [ ]:
# secure_execute: 在隔离进程中执行方法

# 参数:
# worker_execute_method_name: 要执行的方法名
# method_args: 方法位置参数
# method_kwargs: 方法关键字参数
# timeout_seconds: 超时时间
# redirect_to_devnull: 是否丢弃输出

# 返回: ExecutionResults (TypedDict)
# - result: 执行结果
# - execution_time: 执行时间
# - error_msg: 错误信息

print("secure_execute 返回值:")
print("- result: Any - 方法返回的结果")
print("- execution_time: float - 执行时间(秒)")
print("- error_msg: str - 错误信息(如有)")

## 4. 使用示例

In [ ]:
# 示例: 创建一个简单的 Worker 并在 Sandbox 中执行

class SimpleWorker:
    """简单的计算工作器"""
    
    def add(self, a, b):
        """返回两个数的和"""
        return a + b
    
    def compute(self, n):
        """计算 1+2+...+n"""
        return sum(range(1, n+1))

# 创建 worker 和 sandbox
worker = SimpleWorker()
sandbox = SandboxExecutor(
    evaluate_worker=worker,
    join_timeout_seconds=5,
    debug_mode=False
)

# 执行 add 方法
result = sandbox.secure_execute(
    "add",
    method_args=[3, 5],
    timeout_seconds=2
)

print(f"Result: {result['result']}")
print(f"Execution time: {result['execution_time']}")
print(f"Error: {result['error_msg']}")

In [ ]:
# 示例2: 执行带关键字参数的方法

result = sandbox.secure_execute(
    "compute",
    method_kwargs={"n": 100},
    timeout_seconds=2
)

print(f"1+2+...+100 = {result['result']}")

In [ ]:
# 示例3: 处理超时情况

class SlowWorker:
    """模拟耗时操作"""
    
    def slow_task(self, seconds):
        """模拟耗时任务"""
        import time
        time.sleep(seconds)
        return "Done"

slow_worker = SlowWorker()
sandbox = SandboxExecutor(
    evaluate_worker=slow_worker,
    join_timeout_seconds=2  # 2秒超时
)

# 执行需要3秒的任务（会超时）
result = sandbox.secure_execute(
    "slow_task",
    method_args=[3],
    timeout_seconds=2
)

print(f"Result: {result['result']}")
print(f"Error: {result['error_msg']}")
print(f"Execution time: {result['execution_time']}")

In [ ]:
# 示例4: 与 Evaluator 集成

class EvalWorker:
    """评估工作器 - 与 Sandbox 配合使用"""
    
    def __init__(self, test_cases):
        self.test_cases = test_cases
    
    def evaluate(self, program_str):
        """评估程序"""
        try:
            # 模拟评估
            score = 0.95
            return {
                "score": score,
                "execution_time": 0.02,
                "error_msg": None
            }
        except Exception as e:
            return {
                "score": 0.0,
                "execution_time": 0.0,
                "error_msg": str(e)
            }

# 创建工作器和沙箱
test_cases = [([1,2,3], [1,2,3])]
eval_worker = EvalWorker(test_cases)
sandbox = SandboxExecutor(eval_worker, join_timeout_seconds=10)

# 在沙箱中执行评估
result = sandbox.secure_execute(
    "evaluate",
    method_args=["def test(): pass"]
)

print(f"评估结果: {result['result']}")

In [ ]:
# 示例5: 使用 output redirection

class OutputWorker:
    """产生输出的工作器"""
    
    def print_hello(self):
        print("Hello from sandbox!")
        return "Done"

worker = OutputWorker()

# 不重定向输出
result = sandbox.secure_execute(
    "print_hello",
    redirect_to_devnull=False
)
print(f"Result: {result['result']}")

# 重定向输出到 /dev/null
result = sandbox.secure_execute(
    "print_hello",
    redirect_to_devnull=True
)
print(f"Result (output hidden): {result['result']}")

## 总结

| 方法/参数 | 说明 |
|------------|------|
| `SandboxExecutor` | 初始化沙箱 |
| `secure_execute()` | 在隔离进程中执行 |
| `timeout_seconds` | 超时时间 |
| `redirect_to_devnull` | 是否隐藏输出 |
| `ExecutionResults` | 返回结果包含 result, execution_time, error_msg |